In [1]:
from pathlib import Path
import time, json, gc
import numpy as np
import pandas as pd
import joblib
from collections import deque

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FEATURE_DIR = PROJECT_ROOT / "data" / "processed" / "features"
TRAIN_DIR = PROJECT_ROOT / "data" / "processed" / "train"
TEST_DIR = PROJECT_ROOT / "data" / "processed" / "student_test"
MODEL_PATH = PROJECT_ROOT / "models" / "best_candidate_model_bayesian_xgb.joblib"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "viterbi_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = pd.read_csv(FEATURE_DIR / "feature_columns.csv")["feature"].tolist()
model = joblib.load(MODEL_PATH)
train_candidates = pd.read_csv(FEATURE_DIR / "train_candidates.csv")
test_candidates = pd.read_csv(FEATURE_DIR / "student_test_candidates.csv")
for frame in (train_candidates, test_candidates):
    frame["trip_id"] = frame["trip_id"].astype(str).str.zfill(8)
    frame["observation_pos"] = pd.to_numeric(frame["observation_pos"], errors="coerce").astype(int)
    frame["arc_id"] = pd.to_numeric(frame["arc_id"], errors="coerce").astype(int)
    frame[FEATURES] = frame[FEATURES].replace([np.inf, -np.inf], np.nan)

print("Model:", MODEL_PATH.name)
print("Train candidate rows:", len(train_candidates), "| trips:", train_candidates.trip_id.nunique())
print("Student-test candidate rows:", len(test_candidates), "| trips:", test_candidates.trip_id.nunique())

Model: best_candidate_model_bayesian_xgb.joblib
Train candidate rows: 2606160 | trips: 80
Student-test candidate rows: 275425 | trips: 10


In [2]:
def load_graph(trip_dir):
    arcs = pd.read_csv(trip_dir / "arcs.csv")
    # Arc IDs in this dataset are row-position based; preserve row order.
    if "from_node_id" not in arcs or "to_node_id" not in arcs:
        raise ValueError(f"Unexpected arcs.csv schema in {trip_dir}")
    arcs = arcs.reset_index(drop=True)
    return {i: (int(r.from_node_id), int(r.to_node_id)) for i, r in arcs.iterrows()}

def shortest_hops(start, goal, adjacency, max_hops=4):
    if start == goal:
        return 0
    q = deque([(start, 0)])
    seen = {start}
    while q:
        node, depth = q.popleft()
        if depth >= max_hops:
            continue
        for nxt in adjacency.get(node, ()):
            if nxt == goal:
                return depth + 1
            if nxt not in seen:
                seen.add(nxt)
                q.append((nxt, depth + 1))
    return None

def transition_log_score(prev_arc, curr_arc, arc_ends, adjacency, hop_cache):
    if prev_arc == curr_arc:
        return -0.35
    if prev_arc not in arc_ends or curr_arc not in arc_ends:
        return -9.0
    prev_to = arc_ends[prev_arc][1]
    curr_from = arc_ends[curr_arc][0]
    key = (prev_to, curr_from)
    if key not in hop_cache:
        hop_cache[key] = shortest_hops(prev_to, curr_from, adjacency, max_hops=4)
    hops = hop_cache[key]
    if hops is None:
        return -9.0
    return -0.8 - 0.65 * hops

def viterbi_decode(group, arc_ends):
    group = group.sort_values(["observation_pos", "candidate_rank" if "candidate_rank" in group else "arc_id"]).copy()
    observations = [g.reset_index(drop=True) for _, g in group.groupby("observation_pos", sort=True)]
    if not observations:
        return pd.DataFrame()
    adjacency = {}
    for _, (u, v) in arc_ends.items():
        adjacency.setdefault(u, set()).add(v)
    hop_cache = {}
    dp_prev = None
    backpointers = []
    state_ids = []
    for t, obs in enumerate(observations):
        ids = obs.arc_id.astype(int).to_numpy()
        probs = np.clip(obs.emission_probability.to_numpy(dtype=float), 1e-8, 1.0)
        emission = np.log(probs)
        if t == 0:
            dp = emission
            bp = np.full(len(ids), -1, dtype=int)
        else:
            prev_ids = state_ids[-1]
            scores = np.empty((len(prev_ids), len(ids)), dtype=np.float32)
            for i, pa in enumerate(prev_ids):
                for j, ca in enumerate(ids):
                    scores[i, j] = transition_log_score(int(pa), int(ca), arc_ends, adjacency, hop_cache)
            total = dp_prev[:, None] + scores
            bp = np.argmax(total, axis=0)
            dp = total[bp, np.arange(len(ids))] + emission
        state_ids.append(ids)
        backpointers.append(bp)
        dp_prev = dp
    last = int(np.argmax(dp_prev))
    chosen = [last]
    for t in range(len(observations)-1, 0, -1):
        chosen.append(int(backpointers[t][chosen[-1]]))
    chosen.reverse()
    rows = []
    for obs, idx in zip(observations, chosen):
        r = obs.iloc[idx]
        rows.append({"observation_pos": int(r.observation_pos), "arc_id": int(r.arc_id),
                     "emission_probability": float(r.emission_probability),
                     "candidate_rank": int(r.candidate_rank) if "candidate_rank" in r else np.nan})
    return pd.DataFrame(rows)

In [3]:
def decode_partition(frame, graph_root, partition_name):
    all_decoded = []
    summary = []
    for n, (trip_id, candidates) in enumerate(frame.groupby("trip_id", sort=True), 1):
        trip_path = graph_root / trip_id
        if not trip_path.exists():
            print("Skipping missing graph:", trip_id)
            continue
        start = time.perf_counter()
        candidates = candidates.copy()
        candidates["emission_probability"] = model.predict_proba(candidates[FEATURES])[:, 1]
        arc_ends = load_graph(trip_path)
        decoded = viterbi_decode(candidates, arc_ends)
        decoded.insert(0, "trip_id", trip_id)
        decoded.to_csv(OUTPUT_DIR / f"{partition_name}_{trip_id}_viterbi.csv", index=False)
        all_decoded.append(decoded)
        # Simple diagnostics: candidate top-1 vs Viterbi and graph-edge continuity.
        top1 = candidates.sort_values(["observation_pos", "emission_probability"], ascending=[True, False]).drop_duplicates("observation_pos")
        top1_map = top1.set_index("observation_pos").arc_id
        merged = decoded.set_index("observation_pos")
        agreement = float((merged.arc_id == top1_map.reindex(merged.index)).mean())
        ids = decoded.arc_id.astype(int).tolist()
        connected = []
        for a, b in zip(ids[:-1], ids[1:]):
            if a in arc_ends and b in arc_ends:
                connected.append(arc_ends[a][1] == arc_ends[b][0] or a == b)
        summary.append({"trip_id": trip_id, "observations_decoded": len(decoded),
                        "candidate_top1_agreement": agreement,
                        "adjacent_arc_connectivity": float(np.mean(connected)) if connected else np.nan,
                        "runtime_seconds": time.perf_counter()-start})
        print(f"{partition_name} {trip_id}: {len(decoded)} observations ({n}/{frame.trip_id.nunique()})")
        del candidates, decoded
        gc.collect()
    if all_decoded:
        combined = pd.concat(all_decoded, ignore_index=True)
        combined.to_csv(OUTPUT_DIR / f"{partition_name}_viterbi_predictions.csv", index=False)
    summary_df = pd.DataFrame(summary)
    summary_df.to_csv(OUTPUT_DIR / f"{partition_name}_viterbi_summary.csv", index=False)
    return summary_df

train_summary = decode_partition(train_candidates, TRAIN_DIR, "train")
test_summary = decode_partition(test_candidates, TEST_DIR, "student_test")
display(train_summary.head())
display(test_summary)

c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\core.py:569: UserWarning: [15:34:53] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


train 00000001: 1315 observations (1/80)
train 00000002: 7243 observations (2/80)
train 00000003: 821 observations (3/80)
train 00000004: 1064 observations (4/80)
train 00000005: 1045 observations (5/80)
train 00000006: 1450 observations (6/80)
train 00000008: 3093 observations (7/80)
train 00000010: 2711 observations (8/80)
train 00000011: 2828 observations (9/80)
train 00000012: 1608 observations (10/80)
train 00000013: 2580 observations (11/80)
train 00000014: 448 observations (12/80)
train 00000016: 1305 observations (13/80)
train 00000018: 379 observations (14/80)
train 00000019: 2354 observations (15/80)
train 00000020: 2353 observations (16/80)
train 00000021: 15783 observations (17/80)
train 00000022: 474 observations (18/80)
train 00000024: 801 observations (19/80)
train 00000025: 1179 observations (20/80)
train 00000026: 361 observations (21/80)
train 00000027: 694 observations (22/80)
train 00000028: 3565 observations (23/80)
train 00000029: 733 observations (24/80)
train 00

,trip_id,observations_decoded,candidate_top1_agreement,adjacent_arc_connectivity,runtime_seconds
0,00000001,1315,0.938403,0.960426,3.930795
1,00000002,7243,0.679553,0.976664,3.123612
2,00000003,821,0.773447,0.990244,0.721221
3,00000004,1064,0.856203,0.944497,0.603172
4,00000005,1045,0.994258,1.000000,0.282277


,trip_id,observations_decoded,candidate_top1_agreement,adjacent_arc_connectivity,runtime_seconds
0,00000000,2503,0.946864,0.996003,0.924812
1,00000007,1354,0.923191,1.000000,0.390927
2,00000009,1049,0.837941,0.952290,0.678128
3,00000040,2995,0.909182,0.994322,1.224948
4,00000052,262,0.927481,0.965517,0.192009
5,00000056,3958,0.872158,0.967652,2.086690
6,00000065,768,0.855469,0.983051,0.554731
7,00000076,1486,0.814939,0.964310,1.552552
8,00000077,3911,0.719253,0.985934,1.593476
9,00000091,874,0.828375,0.973654,0.567464


In [6]:
if "target" in train_candidates.columns:
    truth = train_candidates.loc[
        train_candidates.target == 1,
        ["trip_id", "observation_pos", "arc_id"]
    ].drop_duplicates(["trip_id", "observation_pos"])

    pred_path = OUTPUT_DIR / "train_viterbi_predictions.csv"

    if pred_path.exists():
        pred = pd.read_csv(pred_path)

        # Make trip IDs and observation positions consistent
        truth["trip_id"] = truth["trip_id"].astype(str).str.zfill(8)
        pred["trip_id"] = pred["trip_id"].astype(str).str.zfill(8)

        truth["observation_pos"] = truth["observation_pos"].astype(int)
        pred["observation_pos"] = pred["observation_pos"].astype(int)

        # Merge predictions with corresponding ground-truth labels
        check = truth.merge(
            pred,
            on=["trip_id", "observation_pos"],
            suffixes=("_true", "_pred")
        )

        check["arc_correct"] = check["arc_id_true"] == check["arc_id_pred"]

        diagnostic = check.groupby("trip_id").arc_correct.mean().reset_index(
            name="viterbi_arc_accuracy"
        )

        diagnostic.to_csv(
            OUTPUT_DIR / "train_viterbi_label_diagnostic.csv",
            index=False
        )

        print("Matched observations:", len(check))
        print("Unmatched truth observations:", len(truth) - len(check))
        print("Overall provisional-label agreement:", round(check.arc_correct.mean(), 4))

        display(diagnostic)

    else:
        print("No combined training predictions found; run section 2 first.")

else:
    print("No target column in training candidates; skipped label diagnostic.")

Matched observations: 206042
Unmatched truth observations: 0
Overall provisional-label agreement: 0.9299


,trip_id,viterbi_arc_accuracy
0,00000001,0.993916
1,00000002,0.786936
2,00000003,0.963459
3,00000004,0.932203
4,00000005,0.974163
...,...,...
75,00000095,0.841393
76,00000096,0.985631
77,00000097,0.994828
78,00000098,0.988181


## 4. Output files and handoff

All outputs are written to `data/processed/viterbi_results/`:

- `train_viterbi_predictions.csv` and one CSV per training trip
- `student_test_viterbi_predictions.csv` and one CSV per student-test trip
- partition summary CSVs with runtime, top-1 agreement, and direct-edge connectivity
- `train_viterbi_label_diagnostic.csv` where training labels are available

The website/API can read these outputs for offline demonstrations. For a live API request, the backend should call the same candidate scoring + Viterbi functions for the selected trip rather than blindly returning a precomputed final answer if the project evaluation requires inference on request.

**Next:** integrate the decoder into the FastAPI backend and expose a `/predict/{trip_id}` endpoint.